# Category Family Extraction

This notebook demonstrates how to extract category families from category codes.
The category family is the first part of the category_code before the dot.

Examples:
- electronics.clock -> electronics
- furniture.table -> furniture
- clothing.shirt -> clothing

In [ ]:
import pandas as pd
import numpy as np
from typing import List, Optional

## Sample Data

Let's create sample data with category codes to demonstrate the extraction.

In [ ]:
# Sample data with category codes
sample_data = {
    'product_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'product_name': [
        'Digital Watch',
        'Office Chair',
        'Winter Jacket',
        'Smartphone',
        'Dining Table',
        'Running Shoes',
        'Laptop Computer',
        'Bookshelf',
        'T-Shirt',
        'Table Lamp'
    ],
    'category_code': [
        'electronics.clock',
        'furniture.chair',
        'clothing.outerwear',
        'electronics.phone',
        'furniture.table',
        'clothing.footwear',
        'electronics.computer',
        'furniture.storage',
        'clothing.shirt',
        'electronics.lighting'
    ],
    'price': [199.99, 249.99, 89.99, 699.99, 449.99, 129.99, 1299.99, 199.99, 29.99, 49.99],
    'quantity': [50, 30, 100, 25, 15, 80, 20, 25, 200, 60]
}

df = pd.DataFrame(sample_data)
print("Sample Data:")
df.head()

## Extract Category Family Function

In [ ]:
def extract_category_family(category_code: str) -> str:
    """
    Extract the category family from a category code.
    The category family is the first part before the dot.
    
    Args:
        category_code: String in format 'family.subcategory' or just 'family'
        
    Returns:
        The category family (first part before the dot)
    """
    if pd.isna(category_code) or category_code == '':
        return 'unknown'
    
    # Split by dot and take the first part
    parts = str(category_code).split('.')
    return parts[0].lower() if parts[0] else 'unknown'

# Test the function
test_codes = [
    'electronics.clock',
    'furniture.chair', 
    'clothing',
    '',
    None,
    'electronics.phone.smartphone'
]

print("Testing category family extraction:")
for code in test_codes:
    family = extract_category_family(code)
    print(f"'{code}' -> '{family}'")

## Apply to DataFrame

In [ ]:
# Apply the function to create category_family column
df['category_family'] = df['category_code'].apply(extract_category_family)

print("DataFrame with category family:")
df[['product_id', 'product_name', 'category_code', 'category_family']].head(10)

## Analyze Category Families

In [ ]:
# Get unique category families
unique_families = df['category_family'].unique()
print(f"Unique category families: {len(unique_families)}")
for family in sorted(unique_families):
    print(f"- {family}")

In [ ]:
# Count products by category family
family_counts = df['category_family'].value_counts().reset_index()
family_counts.columns = ['category_family', 'count']

print("Products count by category family:")
family_counts

## Aggregate by Category Family

In [ ]:
# Aggregate metrics by category family
family_agg = df.groupby('category_family').agg({
    'product_id': 'count',
    'price': ['mean', 'sum', 'min', 'max'],
    'quantity': ['sum', 'mean']
}).round(2)

# Flatten column names
family_agg.columns = ['_'.join(col).strip() for col in family_agg.columns.values]
family_agg = family_agg.reset_index()

print("Aggregated metrics by category family:")
family_agg

## Handle Edge Cases

In [ ]:
# Test edge cases
edge_case_data = {
    'category_code': [
        'electronics',  # No dot
        'electronics.computer.laptop',  # Multiple dots
        '',  # Empty string
        None,  # None value
        'UPPERCASE.CATEGORY',  # Uppercase
        '123.numbers',  # Numbers
        '.leadingdot',  # Leading dot
        'trailingdot.'  # Trailing dot
    ]
}

edge_df = pd.DataFrame(edge_case_data)
edge_df['category_family'] = edge_df['category_code'].apply(extract_category_family)

print("Edge case testing:")
edge_df

## Vectorized Implementation for Large Datasets

In [ ]:
def extract_category_family_vectorized(series: pd.Series) -> pd.Series:
    """
    Vectorized version for better performance on large datasets.
    """
    return series.fillna('unknown').astype(str).str.split('.').str[0].str.lower()

# Compare performance
import time

# Create larger dataset
large_df = pd.DataFrame({
    'category_code': np.random.choice(df['category_code'].tolist(), 10000)
})

# Test original function
start_time = time.time()
result1 = large_df['category_code'].apply(extract_category_family)
time1 = time.time() - start_time

# Test vectorized function
start_time = time.time()
result2 = extract_category_family_vectorized(large_df['category_code'])
time2 = time.time() - start_time

print(f"Original function time: {time1:.4f} seconds")
print(f"Vectorized function time: {time2:.4f} seconds")
print(f"Speed improvement: {time1/time2:.2f}x")
print(f"Results are equal: {result1.equals(result2)}")

## Summary

This notebook demonstrated:
1. How to extract category families from category codes
2. Handling various edge cases (missing values, multiple dots, etc.)
3. Vectorized implementation for better performance
4. Analysis and aggregation by category families

The extracted category families can now be used for:
- Creating embeddings (see category_family_embedding.ipynb)
- Higher-level analysis and reporting
- Feature engineering for machine learning models